## Setup

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import re
import xarray as xr
import glob
from tqdm.notebook import tqdm

In [ ]:
load_dotenv()
raw_path = os.getenv('XRAY_RAW_PATH')
treated_path = os.getenv('XRAY_TREATED_PATH')

data = {}
months = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]

## Read Files

In [ ]:
for y in range(2010, 2024+1):
    year_dir = os.path.join(raw_path, str(y))
    nc_files = glob.glob(os.path.join(year_dir, "*.nc"))

    if not nc_files:
        print(f"Nenhum arquivo encontrado para {y}.")
        continue

    df_days_list = []

    total_rows = 0
    bad_flags_a = 0
    bad_flags_b = 0

    for file_path in tqdm(nc_files, desc=f"Lendo Ano {y}", unit="arq"):
        try:
            with xr.open_dataset(file_path) as ds:
                vars_in_ds = list(ds.variables.keys())

                flag_a_name = 'xrsa_flag' if 'xrsa_flag' in vars_in_ds else 'xrsa_quality'
                flag_b_name = 'xrsb_flag' if 'xrsb_flag' in vars_in_ds else 'xrsb_quality'

                if 'xrsa_flux' not in vars_in_ds or 'xrsb_flux' not in vars_in_ds:
                    continue

                cols_to_extract = ['xrsa_flux', 'xrsb_flux']
                if flag_a_name in vars_in_ds: cols_to_extract.append(flag_a_name)
                if flag_b_name in vars_in_ds: cols_to_extract.append(flag_b_name)

                df_ = ds[cols_to_extract].to_dataframe()

            df_.index.name = 'ds'
            df_ = df_.drop(columns=['Unnamed: 0'], errors='ignore')

            total_rows += len(df_)

            if flag_a_name in df_.columns:
                bad_mask_a = (df_[flag_a_name] != 0) & (df_[flag_a_name].notna())
                bad_flags_a += bad_mask_a.sum()

                df_['xrsa_flux'] = np.where(~bad_mask_a, df_['xrsa_flux'], np.nan)
                df_ = df_.drop(columns=[flag_a_name])

            if flag_b_name in df_.columns:
                bad_mask_b = (df_[flag_b_name] != 0) & (df_[flag_b_name].notna())
                bad_flags_b += bad_mask_b.sum()

                df_['xrsb_flux'] = np.where(~bad_mask_b, df_['xrsb_flux'], np.nan)
                df_ = df_.drop(columns=[flag_b_name])

            if not df_.empty:
                df_days_list.append(df_)
        except Exception as e:
            pass

    if df_days_list:
        df_year = pd.concat(df_days_list).sort_index()
        data[y] = {}
        data[y]["whole_year"] = df_year

        perc_a = (bad_flags_a / total_rows * 100) if total_rows > 0 else 0
        perc_b = (bad_flags_b / total_rows * 100) if total_rows > 0 else 0

        print(f"-> {y} concluido! Shape: {df_year.shape}")
        print(f"   [QUALITY LOG] Dados corrompidos descartados: XRSA: {perc_a:.2f}% | XRSB: {perc_b:.2f}%\n")
    else:
        print(f"-> ALERTA: Erro total ou dados vazios no ano {y}.\n")

print("Leitura NetCDF concluída com sucesso!")

In [ ]:
data[2010]['whole_year']

### Handling Missing Values

In [ ]:
missing_values = [0,-99999,99999]

for y in data.keys():
    df = data[y]["whole_year"]

    df[['xrsa_flux','xrsb_flux']] = df[['xrsa_flux','xrsb_flux']].replace(missing_values, np.nan)
    df[['xrsa_flux', 'xrsb_flux']] = df[['xrsa_flux', 'xrsb_flux']].interpolate(method='time', limit=120)

    missing_xs_count = df['xrsa_flux'].isna().sum()
    missing_xl_count = df['xrsb_flux'].isna().sum()

    rows_count = len(df)
    print(f"{y} Missing Values --> XRSA:{round((missing_xs_count/rows_count)*100,2)}% | XRSB:{round((missing_xl_count/rows_count)*100,2)}%")

In [ ]:
data[2018]['whole_year'][data[2018]['whole_year']['xrsa_flux'].isna()].shape

### Exporting

In [ ]:
# nc_range = range(2020, 2025+1)
# util.create_dirs(treated_path, nc_range)

In [ ]:
# for y in nc_range:
#     df = data[y]["whole_year"]
#
#     year_dir = os.path.join(treated_path, str(y))
#     df.to_csv(os.path.join(year_dir,f"{y}_xrays.csv"))
